In [36]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, UnexpectedAlertPresentException, NoAlertPresentException
import time
import os
import csv
import re

# ==========================================
# 1. 브라우저 및 저장소 설정
# ==========================================

# 폴더명 (원하는 대로 변경 가능)
download_dir = os.path.abspath("downloads_final")
if not os.path.exists(download_dir):
    os.makedirs(download_dir)

prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "profile.default_content_settings.popups": 0,
    "profile.default_content_setting_values.automatic_downloads": 1,
    "plugins.always_open_pdf_externally": True
}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome()
time.sleep(1)
driver.get("https://www.sbiz24.kr/#/combinePbancList")
time.sleep(1)
driver.maximize_window()

# ==========================================
# [함수] 알림창(Alert) 자동 닫기
# ==========================================
def handle_alert():
    """알림창이 떠 있으면 닫고 True 반환, 없으면 False 반환"""
    try:
        # 0.5초만 짧게 확인 (너무 길면 속도 느려짐)
        WebDriverWait(driver, 2).until(EC.alert_is_present())
        alert = driver.switch_to.alert
        print(f"   ⚠️ 알림창 발견(자동 닫기): {alert.text}")
        alert.accept() # 확인(Enter) 누름
        return True
    except:
        return False

# ==========================================
# [함수] 필터 설정 (초기 1회)
# ==========================================
def set_conditions():
    try:
        # 필터창이 없으면 열기
        if not driver.find_elements(By.CSS_SELECTOR, ".modal_style"):
            try:
                WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".f_rcrtTypeCdNmListDisplay button"))).click()
                time.sleep(2)
            except: pass

        # 버튼 클릭 (소상공인 등)
        for idx in [1, 2, 3, 5]:
            try:
                btn = driver.find_element(By.CSS_SELECTOR, f".modal_style .option-area > button:nth-child({idx})")
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.1)
            except: pass

        # 선택완료 & 체크박스
        try:
            driver.execute_script("arguments[0].click();", driver.find_element(By.CSS_SELECTOR, "div.modal_style div.btn-actions > button"))
            time.sleep(1)
            driver.execute_script("arguments[0].click();", driver.find_element(By.CSS_SELECTOR, "#container > div.sub-main-content > div.table-top-area > div > div > label"))
            time.sleep(1)
            print("✅ 초기 조건 설정 완료")
        except: pass
    except Exception as e:
        print(f"조건 설정 중 오류(무시 가능): {e}")

set_conditions()

✅ 초기 조건 설정 완료


In [26]:
def scrape_all_titles(driver):
    """
    1페이지부터 마지막 페이지까지 이동하며 모든 공고의 '제목'을 수집합니다.
    """
    all_titles = []
    page_count = 1
    
    while True:
        print(f"\n📄 {page_count} 페이지 제목 수집 중...")
        
        try:
            # 1. 로딩 대기 (제목 요소가 뜰 때까지)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "td.c_pbancNm > a"))
            )
            
            # 페이지 내의 모든 a 태그 찾기
            elements = driver.find_elements(By.CSS_SELECTOR, "td.c_pbancNm > a")
            
            count_in_page = 0
            for el in elements:
                # -------------------------------------------------
                # [수정된 부분] href 대신 text(보이는 글자) 추출
                # 공백 제거를 위해 .strip() 사용
                # -------------------------------------------------
                title_text = el.text.strip()
                
                # 만약 화면엔 '...'으로 잘려 보이는데 전체 제목이 필요하다면
                # title_text = el.get_attribute("title") # 이 줄의 주석을 푸세요
                
                if title_text:
                    all_titles.append(title_text)
                    count_in_page += 1
            
            print(f"   - {count_in_page}개 제목 수집 완료.")

            # 2. '다음(Next)' 버튼 처리 (기존 로직 동일)
            next_btn_li = driver.find_element(By.CSS_SELECTOR, "ul.pagination li.btn-next")
            next_btn = next_btn_li.find_element(By.TAG_NAME, "button")

            # 3. 마지막 페이지 확인
            is_disabled_li = "disabled" in next_btn_li.get_attribute("class")
            is_disabled_btn = next_btn.get_attribute("disabled") is not None or "disabled" in next_btn.get_attribute("class")
            
            if is_disabled_li or is_disabled_btn:
                print("🏁 마지막 페이지에 도달했습니다.")
                break
            
            # 4. 다음 페이지로 이동
            driver.execute_script("arguments[0].click();", next_btn)
            page_count += 1
            time.sleep(3) 
            
        except TimeoutException:
            print("⚠️ 로딩 시간 초과 또는 데이터 없음. 종료합니다.")
            break
        except Exception as e:
            print(f"⚠️ 에러 발생: {e}")
            break
            
    print(f"\n✅ 총 {len(all_titles)}개의 제목을 수집했습니다.")
    return all_titles

# ==========================================
# 실행 코드
# ==========================================

# 1. 함수 실행
final_titles = scrape_all_titles(driver)

# 2. 결과 출력 (상위 5개만 확인)
print("--- 수집된 제목 예시 ---")
for t in final_titles :
    print(t)

🔎 원본 제목: '[대구] 2025년 4차 소상공인 맞춤형 출산ㆍ양육지원…'
   👉 검색어 변환: '[대구] 2025년 4차 소상공인 맞춤형 출산ㆍ양육지원'
   ⏳ 검색 결과 로딩 중...
   ✅ '[대구] 2025년 4차 소상공인 맞춤형 출산ㆍ양육지원' 관련 결과 1개 발견. 첫 번째 항목 진입.
   🚀 상세 페이지 진입 성공!

[안내] 테스트 완료. 목록으로 돌아가려면 driver.back()을 실행하세요.


In [19]:
driver

165

In [20]:
for i in final_links :
    print(i)

https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117168
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117167
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117098
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117123
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117094
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117103
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117072
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117071
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000117040
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000116998
https://www.sbiz24.kr/#/pbanc/560
https://www.sbiz24.kr/#/pbanc/559
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000116931
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000116894
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000116822
https://www.sbiz24.kr/#/extldPbanc/PBLN_000000000116810
https://www.sbiz24.kr/#/pbanc/553
https://www.sbiz24.kr/#/pbanc/394
https://www.sbiz24.kr/#/pbanc/366
https://www.sbiz24.kr/#/pbanc/387
https://www.

In [21]:
driver.quit()

In [24]:
# 3. CSV로 저장 (제목용으로 수정됨)
import csv
with open("scraped_titles.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["title"]) # 헤더 변경
    for t in final_titles:
        writer.writerow([t])

In [31]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def run_search_navigation_loop(driver, target_titles):
    """
    리스트에 있는 제목을 검색하고, 상세 페이지에 들어갔다가 
    데이터 수집 없이 바로 목록으로 돌아오는 함수입니다.
    """
    print(f"🚀 총 {len(target_titles)}개의 항목을 순회합니다.\n")
    
    for idx, title in enumerate(target_titles):
        print(f"[{idx+1}/{len(target_titles)}] 처리 중...")

        # -------------------------------------------------
        # 1. 검색어 전처리 (말줄임표 제거)
        # -------------------------------------------------
        search_keyword = title
        if "..." in search_keyword: search_keyword = search_keyword.replace("...", "")
        if "…" in search_keyword: search_keyword = search_keyword.replace("…", "")
        search_keyword = search_keyword.strip()
        
        print(f"   🔎 검색: '{search_keyword}'")

        try:
            # -------------------------------------------------
            # 2. 검색 수행
            # -------------------------------------------------
            # 검색창 찾기 (화면 로딩 대기 포함)
            search_input = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "pbancNm"))
            )
            search_input.clear()            # 기존 입력값 지우기 (필수)
            search_input.send_keys(search_keyword) # 키워드 입력
            
            # 조회 버튼 클릭
            search_btn = driver.find_element(By.CSS_SELECTOR, ".btn-box .submit-btn")
            driver.execute_script("arguments[0].click();", search_btn)
            
            # 리스트 갱신 대기 (중요: 너무 빠르면 이전 결과를 클릭함)
            time.sleep(2)

            # -------------------------------------------------
            # 3. 결과 클릭 및 상세 페이지 진입
            # -------------------------------------------------
            results = driver.find_elements(By.CSS_SELECTOR, "td.c_pbancNm > a")
            
            if len(results) > 0:
                # 첫 번째 결과 클릭
                driver.execute_script("arguments[0].click();", results[0])
                
                # 상세 페이지 로딩 대기 (잠깐 들어갔다 나오는 척)
                time.sleep(2) 
                print("   ✅ 상세 페이지 진입 성공")

                # =============================================
                # 여기부터 내용 수집 없이 바로 복귀합니다
                # =============================================
                
                # -------------------------------------------------
                # 4. 목록으로 돌아가기 (뒤로가기)
                # -------------------------------------------------
                driver.back()
                print("   🔙 목록으로 복귀 완료")
                
                # 목록 페이지가 다시 로딩될 때까지 잠시 대기
                time.sleep(2) 
                
            else:
                print("   ⚠️ 검색 결과가 없습니다. (다음 항목으로 이동)")
                # 검색 결과가 없으면 뒤로가기 할 필요 없이 바로 다음 루프로
                
        except Exception as e:
            print(f"   ❌ 에러 발생 (건너뜀): {e}")
            # 에러가 나도 멈추지 않고 다음 걸로 넘어가기 위해 검색창 초기화 시도
            try:
                driver.find_element(By.ID, "pbancNm").clear()
            except:
                driver.refresh() # 심각하면 새로고침
                time.sleep(3)

    print("\n🏁 모든 리스트 순회 완료!")

# ==========================================
# 실행 코드
# ==========================================

# 1. 테스트할 제목 리스트 (앞에서 수집한 final_titles가 있다면 그걸 쓰세요)
if 'final_titles' not in locals() or not final_titles:
    test_titles = [
        "2026년 소공인 특화지원센터 운영기관 모집 공고",
        "2025년 4차 소상공인 맞춤형 출산 · 양육지원 사업..." 
    ]
else:
    test_titles = final_titles

# 2. 함수 실행
run_search_navigation_loop(driver, test_titles)

🚀 총 165개의 항목을 순회합니다.

[1/165] 처리 중...
   🔎 검색: '[대구] 2025년 4차 소상공인 맞춤형 출산ㆍ양육지원'
   ✅ 상세 페이지 진입 성공
   🔙 목록으로 복귀 완료
[2/165] 처리 중...
   🔎 검색: '[경기] 포천시 2026년 소공인가구지원센터 운영 지원'
   ✅ 상세 페이지 진입 성공
   🔙 목록으로 복귀 완료
[3/165] 처리 중...
   🔎 검색: '[부산] 2025년 폐업소상공인 희망두배통장 지원사업('
   ✅ 상세 페이지 진입 성공
   🔙 목록으로 복귀 완료
[4/165] 처리 중...
   🔎 검색: '[충남] 아산시 2025년 4분기분 소상공인 사회보험료'
   ✅ 상세 페이지 진입 성공
   🔙 목록으로 복귀 완료
[5/165] 처리 중...
   🔎 검색: '[충남] 2026년 소상공인자금 융자 지원계획 공고'


KeyboardInterrupt: 

In [37]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, UnexpectedAlertPresentException, NoAlertPresentException
import time
import csv
import os

# ==========================================
# [보조 함수] 알림창 처리
# ==========================================
def handle_alert(driver):
    try:
        WebDriverWait(driver, 1).until(EC.alert_is_present())
        alert = driver.switch_to.alert
        print(f"   ⚠️ 알림창 발견: {alert.text}")
        alert.accept()
        return True
    except:
        return False

# ==========================================
# [메인 함수] 검색 -> 상세수집 -> 뒤로가기
# ==========================================
def run_crawling_search_mode(driver, target_titles):
    print(f"🚀 총 {len(target_titles)}개의 공고 수집을 시작합니다.\n")
    
    final_data = [] # 수집된 데이터를 저장할 리스트
    main_window = driver.current_window_handle # 메인 창 핸들 저장

    for idx, title in enumerate(target_titles):
        print(f"[{idx+1}/{len(target_titles)}] 처리 중...")

        # -------------------------------------------------
        # 1. 검색어 전처리 및 검색 수행 (아래 코드 로직)
        # -------------------------------------------------
        search_keyword = title
        if "..." in search_keyword: search_keyword = search_keyword.replace("...", "")
        if "…" in search_keyword: search_keyword = search_keyword.replace("…", "")
        search_keyword = search_keyword.strip()
        
        print(f"   🔎 검색: '{search_keyword}'")

        try:
            # 검색창 찾기 및 입력
            search_input = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "pbancNm"))
            )
            search_input.clear()
            search_input.send_keys(search_keyword)
            
            # 조회 버튼 클릭
            search_btn = driver.find_element(By.CSS_SELECTOR, ".btn-box .submit-btn")
            driver.execute_script("arguments[0].click();", search_btn)
            time.sleep(2) # 리스트 갱신 대기

            # -------------------------------------------------
            # 2. 결과 클릭 및 상세 페이지 진입
            # -------------------------------------------------
            results = driver.find_elements(By.CSS_SELECTOR, "td.c_pbancNm > a")
            
            if len(results) > 0:
                driver.execute_script("arguments[0].click();", results[0])
                time.sleep(2) # 상세 페이지 로딩 대기

                # =========================================================
                # [3. 상세 내용 수집 및 파일 다운로드] (위의 코드 로직 삽입)
                # =========================================================
                print("   📝 상세 내용 수집 중...")
                
                # 변수 초기화
                title_text = "제목 없음"
                content_body = "본문 없음"
                file_names = []
                current_url = driver.current_url

                try:
                    # (1) 로딩 대기 (제공해주신 선택자 사용)
                    try:
                        WebDriverWait(driver, 10).until(
                            EC.presence_of_element_located((By.CSS_SELECTOR, "#contents > div.u-page.cont-max"))
                        )
                    except TimeoutException:
                        print("      ⚠️ 로딩 시간 초과 (본문 요소 못 찾음)")
                        if handle_alert(driver): pass 
                    
                    # (2) 제목 수집
                    try:
                        title_text = driver.find_element(By.CSS_SELECTOR, "#contents > div.page-header > h2").text
                    except: pass

                    # (3) 본문 수집 (통째로)
                    try:
                        content_body = driver.find_element(By.CSS_SELECTOR, "#contents > div.u-page.cont-max").text
                    except: pass

                    # (4) 파일 다운로드 및 팝업 닫기 (제공해주신 로직 그대로)
                    files = driver.find_elements(By.CSS_SELECTOR, "div.file-group button")
                    
                    if files:
                        print(f"      💾 첨부파일 {len(files)}개 발견. 다운로드 시도...")
                        for f in files:
                            try:
                                f_name = f.text.strip()
                                if not f_name: continue
                                
                                file_names.append(f_name)
                                driver.execute_script("arguments[0].click();", f)
                                time.sleep(3) # 다운로드/새창 대기
                                
                                # 새 창(팝업) 닫기 로직
                                if len(driver.window_handles) > 1:
                                    for handle in driver.window_handles:
                                        if handle != main_window:
                                            driver.switch_to.window(handle)
                                            driver.close()
                                    driver.switch_to.window(main_window)
                            except Exception as e:
                                print(f"      ❌ 파일 클릭 에러: {e}")
                                driver.switch_to.window(main_window)
                    
                    # 수집 데이터 저장
                    final_data.append({
                        "제목": title_text,
                        "검색어": search_keyword,
                        "URL": current_url,
                        "본문": content_body,
                        "첨부파일": ", ".join(file_names)
                    })
                    print("   ✅ 데이터 저장 완료")

                except Exception as e:
                    print(f"   ❌ 상세 수집 중 에러: {e}")

                # -------------------------------------------------
                # 4. 목록으로 돌아가기 (뒤로가기)
                # -------------------------------------------------
                driver.back()
                time.sleep(2) # 목록 로딩 대기
                
            else:
                print("   ⚠️ 검색 결과가 없습니다. (다음 항목으로 이동)")
                
        except Exception as e:
            print(f"   ❌ 프로세스 에러 (건너뜀): {e}")
            # 에러 발생 시 검색창 초기화 또는 새로고침으로 복구 시도
            try:
                driver.find_element(By.ID, "pbancNm").clear()
            except:
                driver.refresh()
                time.sleep(3)

    # -------------------------------------------------
    # 5. CSV 파일 저장
    # -------------------------------------------------
    if final_data:
        csv_filename = "merged_crawling_result.csv"
        with open(csv_filename, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["제목", "검색어", "URL", "본문", "첨부파일"])
            writer.writeheader()
            writer.writerows(final_data)
        print(f"\n🎉 작업 완료! '{csv_filename}' 파일이 생성되었습니다.")
    else:
        print("\n⚠️ 수집된 데이터가 없습니다.")

# ==========================================
# 실행 코드
# ==========================================

# 1. 수집할 제목 리스트 준비
# (final_titles 변수가 있다면 사용, 없으면 테스트용 리스트 사용)
if 'final_titles' not in locals() or not final_titles:
    print("⚠️ 제목 리스트가 없어 테스트 데이터를 사용합니다.")
    target_list = [
        "2026년 소공인 특화지원센터 운영기관 모집 공고", # 실제 존재하는 제목 예시
        "테스트용 없는 제목 1234"
    ]
else:
    target_list = final_titles

# 2. 함수 실행
run_crawling_search_mode(driver, target_list)

🚀 총 165개의 공고 수집을 시작합니다.

[1/165] 처리 중...
   🔎 검색: '[대구] 2025년 4차 소상공인 맞춤형 출산ㆍ양육지원'
   📝 상세 내용 수집 중...
      💾 첨부파일 4개 발견. 다운로드 시도...
   ✅ 데이터 저장 완료
[2/165] 처리 중...
   🔎 검색: '[경기] 포천시 2026년 소공인가구지원센터 운영 지원'
   📝 상세 내용 수집 중...
      💾 첨부파일 1개 발견. 다운로드 시도...
   ✅ 데이터 저장 완료
[3/165] 처리 중...
   🔎 검색: '[부산] 2025년 폐업소상공인 희망두배통장 지원사업('
   📝 상세 내용 수집 중...
      💾 첨부파일 5개 발견. 다운로드 시도...
   ✅ 데이터 저장 완료
[4/165] 처리 중...
   🔎 검색: '[충남] 아산시 2025년 4분기분 소상공인 사회보험료'
   📝 상세 내용 수집 중...
      💾 첨부파일 2개 발견. 다운로드 시도...
   ✅ 데이터 저장 완료
[5/165] 처리 중...
   🔎 검색: '[충남] 2026년 소상공인자금 융자 지원계획 공고'
   📝 상세 내용 수집 중...
      💾 첨부파일 1개 발견. 다운로드 시도...
   ✅ 데이터 저장 완료
[6/165] 처리 중...
   🔎 검색: '[대전] 유성구 2026년 소상공인 특례보증 지원사업'
   📝 상세 내용 수집 중...
      💾 첨부파일 1개 발견. 다운로드 시도...
   ✅ 데이터 저장 완료
[7/165] 처리 중...
   🔎 검색: '[경기] 부천시 2026년 골목형상점가 지정 모집 공고'
   📝 상세 내용 수집 중...
      💾 첨부파일 1개 발견. 다운로드 시도...
   ✅ 데이터 저장 완료
[8/165] 처리 중...
   🔎 검색: '[부산] 사상구 2026년 소상공인 보증료 지원사업 대'
   📝 상세 내용 수집 중...
      💾 첨부파일 1개 발견. 다운로드 시도...
   ✅ 데이터 

In [32]:
driver.quit()